# 02 — Texture Classification with ResNet-50
Train and evaluate the ResNet-50 texture classifier on DTD.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

In [ ]:
# ── Build data loaders ──
from data.texture_dataset import build_dtd_loaders, DTDDataset

DTD_ROOT = '../data/dtd'
train_loader, val_loader, test_loader = build_dtd_loaders(DTD_ROOT, batch_size=32)
print(f'Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)} | Test: {len(test_loader.dataset)}')
print(f'Classes: {DTDDataset.get_class_names()[:5]} ...')

In [ ]:
# ── Visualize batch ──
from torchvision.utils import make_grid

images, labels = next(iter(train_loader))
class_names = DTDDataset.get_class_names()

mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
imgs_denorm = (images[:16] * std + mean).clamp(0, 1)

grid = make_grid(imgs_denorm, nrow=8, padding=4)
plt.figure(figsize=(16, 5))
plt.imshow(grid.permute(1,2,0))
plt.title('Training Batch Sample  |  ' + '  '.join([class_names[l] for l in labels[:8]]))
plt.axis('off'); plt.tight_layout(); plt.show()

In [ ]:
# ── Initialize model ──
from models.classifier import TextureClassifier

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TextureClassifier(pretrained=True).to(device)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total:,}  |  Trainable: {trainable:,}  ({trainable/total*100:.1f}%)')

In [ ]:
# ── Quick training run (5 epochs demo) ──
# For full training: python training/train_classifier.py --config configs/classifier.yaml
import torch.nn as nn
from torch.optim import AdamW

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

train_accs, val_accs = [], []
for epoch in range(1, 4):
    model.train()
    correct, total = 0, 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), lbls)
        loss.backward(); optimizer.step()
        correct += (model(imgs).argmax(1) == lbls).sum().item()
        total += lbls.size(0)
        break  # 1 batch per epoch for demo
    
    train_acc = correct / total
    train_accs.append(train_acc)
    print(f'Epoch {epoch} | train_acc={train_acc:.3f}')

In [ ]:
# ── Feature embedding visualization with t-SNE ──
from sklearn.manifold import TSNE

model.eval()
feats, lbls_all = [], []
with torch.no_grad():
    for imgs, lbls in test_loader:
        f = model.get_feature_vector(imgs.to(device)).cpu().numpy()
        feats.append(f); lbls_all.extend(lbls.tolist())
        if len(lbls_all) >= 500: break

feats = np.concatenate(feats, axis=0)[:500]
lbls_all = lbls_all[:500]

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
emb = tsne.fit_transform(feats)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(emb[:,0], emb[:,1], c=lbls_all, cmap='tab20', s=8, alpha=0.7)
plt.colorbar(scatter, label='Texture class')
plt.title('t-SNE of ResNet-50 texture features (500 samples)')
plt.tight_layout(); plt.show()